# AutoSort Training Pipeline

Main steps:
1. Threshold detection
2. Training data preparation
3. Model training


In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

from pathlib import Path
from utils_clean import (
    prepare_training_data,
    train_autosort_model
)


In [ ]:
# Load data
recording_path = '/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_021322_natural_image_001.ns4'
spike_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/021322/spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/021322/neuron_inf.pkl"

# Load GT data
spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
with open(neuron_inf_path, 'rb') as f:
    neuron_inf = pickle.load(f)

# Load and preprocess recording
recording_raw = se.read_blackrock(file_path=recording_path)
recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")

print(f"Recording loaded successfully")
print(f"Sampling rate: {recording_f.get_sampling_frequency()} Hz")
print(f"Number of channels: {recording_f.get_num_channels()}")


Recording 加载完成
采样率: 10000.0 Hz
通道数: 30


## Step 1: Threshold Detection + Training Data Preparation


In [3]:
# Set parameters
save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/"
duration_seconds = 200  # Processing duration (seconds)

# Extract all unique tract_channels from neuron_inf for threshold detection on these channels only
valid_channels = sorted(neuron_inf['tract_channel'].unique().tolist())
print(f"Number of valid channels extracted from neuron_inf: {len(valid_channels)}")
print(f"Valid channels list: {valid_channels}")

# Detection parameters (consistent with AutoSort default values)
detection_params = {
    'thr_min': 3.5,
    'thr_max': 30,
    'distance': 3,
    'ch_max_simul_firing': 5,
    'wlen': 5,
    'prominence': 10,
}

# Waveform window parameters
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Prepare training data (includes threshold detection, GT matching, waveform extraction, data saving)
train_data_dir = prepare_training_data(
    recording_f=recording_f,
    spike_inf=spike_inf,
    neuron_inf=neuron_inf,
    save_dir=save_dir,
    duration_seconds=duration_seconds,
    valid_channels=valid_channels,  # Pass valid_channels parameter to detect only on valid channels
    **detection_params,
    **window_params
)


### 1. 阈值检测
采样率: 10000.0 Hz, 通道数: 30
Recording总长度: 40000100 采样点 (4000.01 秒)
将处理前 2000000 采样点 (200.00 秒)
数据形状: (2000000, 30)
构建 detect_array...
检测到的 spike 数量: 431532

### 2. 加载 Ground Truth 并匹配
构建 gt_array...
GT spike 数量: 74790
---spike detection rate: 0.9266
匹配到的 spike 数量: 69300
未匹配的 spike 数量: 362232

### 3. 提取波形


提取波形: 100%|██████████| 30/30 [00:08<00:00,  3.41it/s]


波形提取完成！
waveform 形状: (431525, 30, 30)

### 4. 保存训练数据
保存目录: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/train_data
  ✓ neuron_mapping.pkl 已保存
保存数据...
  ✓ X_waveform.pkl 已保存
  ✓ Y_spike_id.pkl 已保存
  ✓ Y_spike_id_noise.pkl 已保存
  ✓ X_spiketrain_time.pkl 已保存

所有数据已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/train_data
数据统计:
  - 总 spike 数量: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声 spike 数量: 362227
  - 有效 spike 数量: 69298


## Step 2: Model Training


In [4]:
# Set training parameters
base_model_save_dir = save_dir + "model_save/"
n_channels = recording_f.get_num_channels()

training_params = {
    'epochs': 20,
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
    'early_stopping': True,  # Enable early stopping
    'patience': 5,  # Stop if accuracy doesn't improve for 5 consecutive epochs
    'min_delta': 0.0,  # Minimum change
}

# Repeat training 5 times
n_runs = 5
all_models = []
all_logs = []

for run_id in range(1, n_runs + 1):
    print(f"\n{'='*60}")
    print(f"Starting training run {run_id}/{n_runs}")
    print(f"{'='*60}")
    
    # Create independent save directory for each training run
    model_save_dir = base_model_save_dir + f"run_{run_id}/"
    
    # Train model
    autosort_model, training_log = train_autosort_model(
        train_data_dir=train_data_dir,
        model_save_dir=model_save_dir,
        n_channels=n_channels,
        **training_params
    )
    
    all_models.append(autosort_model)
    all_logs.append(training_log)
    
    print(f"\nTraining run {run_id} completed!")
    print(f"Model save directory: {model_save_dir}")

print(f"\n{'='*60}")
print(f"All {n_runs} training runs completed!")
print(f"{'='*60}")



开始第 1/5 次训练
使用设备: cuda
创建 dataset...
Dataset 加载完成:
  - 总样本数: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声样本数: 362227.0
  - 非噪声样本数: 69298.0
模型参数:
  - 通道数: 30
  - 窗口长度: 30
  - 单元数量: 27
  - 输入维度: 930
单元ID列表已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/run_1/keep_id.pkl

数据集划分:
  - 训练集: 345220 样本
  - 验证集: 86305 样本
已加载现有模型

第 1 次训练完成！
模型保存目录: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/run_1/

开始第 2/5 次训练
使用设备: cuda
创建 dataset...
Dataset 加载完成:
  - 总样本数: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声样本数: 362227.0
  - 非噪声样本数: 69298.0
模型参数:
  - 通道数: 30
  - 窗口长度: 30
  - 单元数量: 27
  - 输入维度: 930
单元ID列表已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/run_2/keep_id.pkl

数据集划分:
  - 训练集: 345220 样本
  - 验证集: 86305 样本
已加载现有模型

第 2 次训练完成！
模型保存目录: /media/ubuntu/s